# Session 7 — t-Test

**Goal:** run Session 6's framework on the most common concrete question in this
registry — *is a continuous measurement different between two groups?* — with the
assumption checks, the variance decision, the confidence interval, the effect size, and
the fallback for when the assumptions genuinely fail.

## What this stage does for the system

Session 6 established the framework and used a proportion comparison to demonstrate it.
Most of the interesting inputs here are not proportions: `thalach`, `oldpeak`, `age`,
`trestbps` are continuous, and the question about each is the same — does it separate
diseased from non-diseased patients by more than sampling noise?

That question has three answers depending on choices you make along the way, and this
session is about making them deliberately:

1. **Equal variances or not?** Student's classic t-test assumes the two groups share a
   variance; Welch's does not. Getting this wrong inflates the false-positive rate.
2. **Paired or independent?** Two measurements *on the same patient* carry information
   the independent test throws away. Step 8 shows the same effect going from
   undetectable to overwhelming purely on this choice.
3. **Normal enough?** When it is not, a rank-based test answers the question without
   the assumption.

Every "is this input worth keeping" decision that reaches Session 9 passes through one
of these tests, so the answers are load-bearing.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — The comparison, stated before testing

`thalach` (maximum heart rate achieved on a stress test) had the strongest continuous
correlation with disease in Session 5. The hypotheses, following Session 6's ordering:

- $H_0$: mean `thalach` is equal in the two outcome groups.
- $H_1$: the means differ (two-sided).
- $\alpha = 0.05$.

In [ ]:
ALPHA = 0.05

disease = df.loc[df["target"] == 1, "thalach"]
no_disease = df.loc[df["target"] == 0, "thalach"]

print(f"{'group':12} {'n':>5} {'mean':>8} {'sd':>8}")
print(f"{'disease':12} {len(disease):>5} {disease.mean():>8.2f} {disease.std():>8.2f}")
print(f"{'no disease':12} {len(no_disease):>5} {no_disease.mean():>8.2f} {no_disease.std():>8.2f}")
print(f"\nobserved difference: {disease.mean() - no_disease.mean():+.2f} bpm")

**Observe:** 137 diseased patients averaging `139.11` bpm against 160 non-diseased
averaging `158.58` — a gap of `−19.47` bpm — and standard deviations that are *not*
equal: `22.71` versus `19.04`.
**Infer:** the direction is clinically sensible (diseased patients cannot push their
heart rate as high under stress), which is a weak but real check that the outcome
coding has not been flipped somewhere upstream. The unequal standard deviations are the
consequential detail: the diseased group is both lower *and* more spread out, which is
what Step 4's variance decision hinges on. Note it is more than a nuisance — a group
that is more variable is intrinsically harder to predict, which reappears in Session
12 as a floor on achievable accuracy.

## Step 3 — Check the assumptions before trusting the test

The t-test assumes each group's values are approximately Normal (or that the samples
are large enough for the CLT to carry the *sampling distribution* of the mean, per
Session 4). Shapiro-Wilk tests that group by group — with Session 3's caveat that on
297 patients it will reject deviations too small to matter, so the plot decides.

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

for name, group in [("disease", disease), ("no disease", no_disease)]:
    result = stats.shapiro(group)
    print(f"Shapiro-Wilk, {name:11} p={result.pvalue:.4f}  "
          f"skew={stats.skew(group):+.2f}  "
          f"-> {'Normality rejected' if result.pvalue < ALPHA else 'no evidence against Normality'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist([no_disease, disease], bins=20, label=["no disease", "disease"],
             color=["steelblue", "indianred"])
axes[0].set_xlabel("thalach (bpm)"); axes[0].set_ylabel("patients")
axes[0].set_title("Distribution by outcome group"); axes[0].legend()

axes[1].boxplot([no_disease, disease], tick_labels=["no disease", "disease"])
axes[1].set_ylabel("thalach (bpm)")
axes[1].set_title("Same data, spread and outliers")
plt.tight_layout()
plt.show()

**Observe:** the disease group passes comfortably (`p = 0.405`) while the no-disease
group is rejected (`p = 0.0010`, skew `−0.69`), and the boxplot shows the no-disease
group tighter with a low-side tail.
**Infer:** one group Normal and the other not is the ordinary situation, and it does not
block the test. At n = 137 and n = 160 the Central Limit Theorem is doing the work: the
t-test needs the *sampling distribution of each mean* to be Normal, not the raw values,
and Session 4 showed that distribution converging fast even from a strongly skewed
column. Moderate skew at this sample size is fine; the cases to worry about are small
groups (under ~30) or extreme skew, and Step 7 keeps a distribution-free option ready
for exactly those. Notice the checks disagree usefully with each other — Shapiro
rejects while the histogram looks unremarkable, which is Session 3's point about
Normality tests on large samples, restated on new data.

## Step 4 — Welch's t-test, and why it is the safer default

Student's classic t-test pools both groups into one variance estimate — valid only if
they really do share one. **Welch's** t-test estimates each group's variance separately
and adjusts the degrees of freedom accordingly. It costs essentially nothing when
variances *are* equal, and it protects you when they are not.

In [ ]:
levene = stats.levene(disease, no_disease)
print(f"Levene's test for equal variances: p={levene.pvalue:.4f}  "
      f"-> {'variances differ' if levene.pvalue < ALPHA else 'no evidence they differ'}")
print(f"  sd ratio: {disease.std() / no_disease.std():.3f}\n")

welch = stats.ttest_ind(disease, no_disease, equal_var=False)
classic = stats.ttest_ind(disease, no_disease, equal_var=True)

print(f"Welch's t-test:  t={welch.statistic:.3f}, p={welch.pvalue:.2e}, df={welch.df:.1f}")
print(f"Classic t-test:  t={classic.statistic:.3f}, p={classic.pvalue:.2e}, df={classic.df:.1f}")
print(f"\nreject H0 at alpha={ALPHA}? Welch: {welch.pvalue < ALPHA}, classic: {classic.pvalue < ALPHA}")

**Observe:** Levene rejects equal variances (`p = 0.014`), and the two tests give
`p = 6.1e-14` (Welch) versus `2.2e-14` (classic) — the classic test's degrees of
freedom are the full `295.0` while Welch's are reduced to `266.4`.
**Infer:** both reject overwhelmingly, so the choice changes nothing *here* — but that
is a property of a huge effect, not a general reassurance. The classic test reports a
smaller p-value because it claims more degrees of freedom than the data supports, and
near the 0.05 boundary that difference decides the outcome. Since Welch costs almost
nothing when variances match, "always use Welch" is a better rule than "test for equal
variances, then choose" — a two-stage procedure has its own false-positive rate, and
Levene's test is itself underpowered on small groups, exactly where the choice matters
most. `scipy` unfortunately defaults to `equal_var=True`, so this is a flag to pass
explicitly every time.

## Step 5 — The confidence interval for the difference

The p-value says the gap is not zero. The interval says how big it plausibly is —
which is the version a clinician can use.

In [ ]:
import numpy as np

diff = disease.mean() - no_disease.mean()
se = np.sqrt(disease.var(ddof=1) / len(disease) + no_disease.var(ddof=1) / len(no_disease))
t_crit = stats.t.ppf(0.975, df=welch.df)

low, high = diff - t_crit * se, diff + t_crit * se
print(f"difference in means: {diff:+.2f} bpm")
print(f"standard error:       {se:.2f}")
print(f"95% CI: [{low:.2f}, {high:.2f}] bpm")
print(f"\ninterval excludes zero? {not (low <= 0 <= high)}  (equivalent to p < {ALPHA})")

**Observe:** `[−24.31, −14.64]` bpm — a wide interval, entirely below zero.
**Infer:** "somewhere between 15 and 24 bpm lower" is a far more useful sentence than
"p = 6.1e-14", because it is stated in the units the measurement was taken in and it
carries its own uncertainty. The interval and the test are the same computation seen
from two directions — an interval excluding zero is exactly a rejection at
$\alpha = 0.05$ — so reporting the interval strictly dominates reporting the p-value:
it answers the significance question *and* the magnitude question. Note the width, ten
bpm, is what 297 patients buys; Session 4's $\sqrt{n}$ says quartering it needs
sixteen times the registry.

## Step 6 — Effect size: Cohen's d

The interval is in bpm, which does not compare across inputs measured in different
units. **Cohen's d** expresses the gap in pooled standard deviations: 0.2 small, 0.5
medium, 0.8 large.

In [ ]:
def cohens_d(a, b):
    pooled_sd = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
                        / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / pooled_sd

print(f"{'input':10} {'diff':>9} {'Cohen d':>9} {'p (Welch)':>12}")
for col in ["thalach", "oldpeak", "age", "trestbps", "chol"]:
    a = df.loc[df["target"] == 1, col]
    b = df.loc[df["target"] == 0, col]
    t = stats.ttest_ind(a, b, equal_var=False)
    print(f"{col:10} {a.mean() - b.mean():+9.2f} {cohens_d(a, b):+9.3f} {t.pvalue:>12.2e}")

**Observe:** `thalach` at `d = −0.935` and `oldpeak` at `+0.936` are large and nearly
mirror images; `age` is moderate; `trestbps` small; `chol` at `d = +0.161` fails to
reach significance at all.
**Infer:** the d column ranks the inputs on the same scale regardless of units, and it
reproduces the ordering Session 5's correlations gave and Session 2's group gaps gave
before that — three different methods agreeing is worth more than any one of them. The
`chol` row closes a thread running since Session 1: weak conditional probability, weak
correlation, and now formally non-significant with a small effect size. It is not a
useful single predictor in this registry. What none of these three methods can say is
whether `chol` matters *in combination* with other inputs, which is the question only
Session 9's multivariate fit can answer — so "drop it" is still premature.

## Step 7 — When the assumptions genuinely fail: Mann-Whitney U

The **Mann-Whitney U** test compares two groups by ranks. It assumes no distribution at
all, testing whether a random value from one group tends to exceed one from the other.
`oldpeak` — Session 3's column that no standard distribution fits — is the natural test
case.

In [ ]:
for col in ["thalach", "oldpeak"]:
    a = df.loc[df["target"] == 1, col]
    b = df.loc[df["target"] == 0, col]
    welch_c = stats.ttest_ind(a, b, equal_var=False)
    mwu = stats.mannwhitneyu(a, b, alternative="two-sided")
    print(f"{col}:")
    print(f"  Shapiro (disease group): p={stats.shapiro(a).pvalue:.2e}")
    print(f"  Welch's t-test:  p={welch_c.pvalue:.2e}")
    print(f"  Mann-Whitney U:  p={mwu.pvalue:.2e}")
    print(f"  zeros in column: {(df[col] == 0).sum()}")

**Observe:** for both columns the two tests agree to within an order of magnitude on
p-values that are astronomically small — and `oldpeak` contains 96 exact zeros, which
is why no continuous distribution fits it.
**Infer:** agreement is the reassuring outcome: it means the t-test's Normality
assumption was not doing any load-bearing work, so its conclusion stands even though
the assumption is violated. That is the right way to use a non-parametric test — as a
*robustness check* alongside the parametric one, not as a replacement. Had they
disagreed, the Mann-Whitney result would be the one to trust for `oldpeak`. The cost of
switching is interpretability: Mann-Whitney does not produce a difference in means or a
confidence interval in the original units, so Step 5's clinically usable sentence is
not available from it. Prefer the parametric test when its assumptions hold; keep this
one for when they do not.

## Step 8 — Paired vs. independent: the same data, two verdicts

If two measurements come from the *same* patient (before and after treatment, or two
visits), they are **paired**, and the paired t-test works on the per-patient
differences — cancelling out between-patient variation entirely. This registry has no
repeated measurements, so the demonstration below is explicitly synthetic; the
mechanism is what matters, and it applies directly the moment follow-up visits exist.

In [ ]:
rng = np.random.default_rng(3)
n = 40

baseline = rng.normal(150, 20, n)            # patients differ a lot from each other
true_effect = -6.0                           # treatment lowers thalach by 6 bpm
followup = baseline + true_effect + rng.normal(0, 5, n)   # small within-patient noise

paired = stats.ttest_rel(followup, baseline)
independent = stats.ttest_ind(followup, baseline, equal_var=False)

print(f"true effect built into the data: {true_effect:+.1f} bpm")
print(f"observed mean difference:        {(followup - baseline).mean():+.2f} bpm\n")
print(f"paired t-test (correct):      t={paired.statistic:7.3f}, p={paired.pvalue:.2e}  -> detected")
print(f"independent t-test (wrong):   t={independent.statistic:7.3f}, p={independent.pvalue:.3f}  -> missed")
print()
print(f"between-patient sd (the noise the paired test removes): {baseline.std(ddof=1):.1f} bpm")
print(f"within-patient sd of the differences:                   {(followup - baseline).std(ddof=1):.1f} bpm")

**Observe:** a real 6 bpm effect that the paired test finds at `p = 3.2e-10` and the
independent test misses entirely at `p = 0.209` — on identical numbers.
**Infer:** the two standard deviations at the bottom explain the whole gap: the
independent test has to see a 6 bpm shift through `23.5` bpm of between-patient
variation, while the paired test differences that variation away and only faces the
`5.1` bpm within-patient noise. Same data, more than four times the signal-to-noise. The general principle:
*pairing is information, and analysing paired data as independent throws it away*. The
error runs in the other direction too and is worse — treating genuinely independent
measurements as paired invents a correlation that is not there and inflates the
false-positive rate. Neither is detectable from the numbers alone; only knowing how the
data was collected tells you which test applies, which is why Session 4's collection
design keeps mattering long after collection is finished.

## What this session hands to the next one

- **A tested set of continuous predictors** with p-values, confidence intervals, and
  effect sizes: `thalach` and `oldpeak` large, `age` moderate, `trestbps` small,
  `chol` not significant.
- **Welch as the default**, and the habit of checking group variances rather than
  assuming they match.
- **Effect size alongside every p-value**, per Session 6.
- **A robustness check** (Mann-Whitney) for columns whose distribution misbehaves.

Session 8 asks the same question for the *categorical* inputs — `cp`, `thal`, `slope`,
`restecg` — where means are meaningless and the comparison is between whole
distributions across categories.

## Try it yourself

1. Re-run Steps 2-6 comparing `thalach` between sexes instead of outcome groups. Does
   Levene still reject equal variances, and does the choice of test matter more there?
2. Subsample each outcome group to 25 patients and re-run Step 4. Which conclusions
   survive, and what does that say about Session 6's power discussion?
3. In Step 7, apply Session 3's `log` transform to `chol` before the t-test. Does
   making the column Normal change the verdict?
4. In Step 8, raise the within-patient noise from 5 to 25 while keeping the effect at
   −6. At what noise level does the paired test's advantage disappear, and why?